# CV Masterclass 08: Generative Vision Lineage

Welcome to Notebook 08. This notebook traces the complete historical and mathematical lineage of **Generative Computer Vision**.

We do not jump straight to the state-of-the-art. To understand *why* Diffusion Models exist, you must understand the mathematical collapse of GANs. To understand GANs, you must understand the blurriness of VAEs. To understand VAEs, you must understand the deterministic failure of standard Autoencoders.

---

## 🎓 Socratic Deep Check
Ponder these questions as we progress through the lineage:

> *"A standard Autoencoder compresses a dog into a latent vector `[2.5, -1.0]`, and successfully reconstructs it. But if you pass the random vector `[2.4, -0.9]` into the Decoder, it outputs absolute garbage instead of a slightly different dog. Why does the math of a Variational Autoencoder (VAE) specifically fix this, allowing us to safely generate new images from random vectors?"*

> *"Stable Diffusion uses a VAE to compress images before diffusing them. The VAE was trained separately with a perceptual loss (LPIPS) to preserve fine visual details. Some experiments showed that using a VAE with a HIGHER compression ratio (e.g., 8x8x4 instead of 64x64x4 = 64x) allowed faster diffusion but produced images with structural artifacts that neither the diffusion model nor the VAE decoder could repair. Why are these structural artifacts specifically IMPOSSIBLE to fix at the decoder stage, given the mathematical nature of the information bottleneck?"*

---

## Table of Contents
1. **The Foundation:** Standard Autoencoders (Compression vs Generation).
2. **The Probabilistic Shift:** Variational Autoencoders (VAEs).
3. **The Adversarial Era:** GANs & WGANs.
4. **StyleGAN:** Disentangled Latent Spaces.
5. **Diffusion Models:** DDPMs and stable Markov Chains.
6. **Latent Diffusion:** Stable Diffusion Architecture & Cross-Attention.
7. **Classifier-Free Guidance (CFG):** Steering the generation.
8. **DDIM:** Deterministic Sampling.
9. **Evaluation Metrics:** FID, LPIPS, and CLIP scores.


## 1. Standard Autoencoders (The Deterministic Bottleneck)

An Autoencoder (AE) has two parts:
1.  **Encoder:** Compresses an $H \times W \times C$ image into a tiny 1-Dimensional Latent Vector (e.g., $Z$, length 128).
2.  **Decoder:** Expands the 128-dim vector back into the original $H \times W \times C$ image.

**Loss:** Mean Squared Error (MSE) between the input image and the output image.

**The Generative Failure:** AEs are deterministic. The network learns to map exactly 1 specific training image to exactly 1 specific point in the 128-D space. The space between the points is completely "empty" or discontinuous. If you sample a random point in that space (trying to *generate* a new image), the decoder panics and outputs meaningless noise. AEs are purely for compression, not generation.


In [ ]:
import torch
import torch.nn as nn
import numpy as np

class SimpleAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(784, 128)
        self.decoder = nn.Linear(128, 784)
        
    def forward(self, x):
        return self.decoder(torch.relu(self.encoder(x)))

# TEST IT
model = SimpleAutoencoder()
dummy_img = torch.randn(1, 784)
out = model(dummy_img)
print(f"AE Output shape: {out.shape}")


### COMMON PITFALLS
*   **Confusing Compression with Generation:** Trying to sample randomly from a standard AE latent space. It will result in severe visual artifacts because the manifold of real images does not smoothly span the latent space.


## 2. Variational Autoencoders (VAEs)

A VAE forces the Latent Space to be continuous. 
Instead of the Encoder outputting a single point $Z$, it outputs two vectors:
1.  **Mean ($\mu$):** The center of a distribution.
2.  **Standard Deviation ($\sigma$):** The spread of the distribution.

We then sample a random point from $\mathcal{N}(\mu, \sigma^2)$ and pass *that* to the Decoder. 

Because the decoder is constantly being fed slightly different points from the distribution during training, it is forced to learn that *entire region* corresponds to a concept. The space becomes perfectly smooth and continuous.

### The Reparameterization Trick
If you have a operation `z = np.random.normal(mu, sigma)`, PyTorch cannot backpropagate through it. The computation graph breaks.
**The Math Fix:** We sample from a standard Normal Distribution $\epsilon \sim \mathcal{N}(0, 1)$ (which requires no gradients), and mathematically shift it: 
$$Z = \mu + \sigma \odot \epsilon$$
Backprop flows perfectly through $\mu$ and $\sigma$!


In [ ]:
# IMPLEMENTATION: The VAE Reparameterization Trick

class VAE_Bottleneck(nn.Module):
    def __init__(self, in_features=512, latent_dim=128):
        super(VAE_Bottleneck, self).__init__()
        self.fc_mu = nn.Linear(in_features, latent_dim)
        self.fc_logvar = nn.Linear(in_features, latent_dim)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)  # N(0, 1) without gradients
        z = mu + (std * eps)
        return z
        
    def forward(self, x):
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

# TEST IT
vae_neck = VAE_Bottleneck()
latent_rep, mu, logvar = vae_neck(torch.randn(2, 512))
print(f"VAE Z shape: {latent_rep.shape}")


### COMMON PITFALLS
*   **Posterior Collapse:** If the KL Divergence penalty is weighted too strongly compared to the reconstruction loss, the encoder will just output $\mu=0, \sigma=1$ for every image. The latent space contains no information.
*   **Blurriness:** VAEs use purely L2 MSE loss for reconstructions across probability distributions, which mathematically averages uncertainty into blurry pixels.


## 3. Generative Adversarial Networks (GANs)

To fix VAE blurriness, Ian Goodfellow invented the GAN (2014). We implement a **Zero-Sum Minimax Game**:
1.  **Generator ($G$):** Takes a random noise vector $Z$ and tries to create crisp pixels to fool $D$.
2.  **Discriminator ($D$):** A binary classifier trying to tell Real images from Fake.

**Mode Collapse:** If $G$ discovers it generates highly realistic "Cats", $D$ assigns them "Real". Because $G$'s only objective is to fool $D$, $G$ stops trying to generate Dogs or Cars entirely. Furthermore, if $D$ gets too good too fast, its gradients drop to $0.0$, and $G$ stops learning completely.


In [ ]:
# IMPLEMENTATION: The WGAN Gradient Penalty (Fixing GANs)

def compute_gradient_penalty(critic, real_samples, fake_samples):
    alpha = torch.rand((real_samples.size(0), 1), device=real_samples.device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = critic(interpolates)
    
    fake = torch.ones(d_interpolates.shape, requires_grad=False, device=real_samples.device)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

# TEST IT
class DummyCritic(nn.Module):
    def forward(self, x): return x.mean(dim=1, keepdim=True)

critic = DummyCritic()
real = torch.randn(4, 128)
fake = torch.randn(4, 128)
gp = compute_gradient_penalty(critic, real, fake)
print(f"Gradient Penalty magnitude: {gp.item():.4f}")


### COMMON PITFALLS
*   **Too strong Discriminator:** If the discriminator dominates early on, it will push the generator's gradients to zero in standard GANs. WGAN-GP relies on the Lipschitz constraint to ensure gradients are always meaningful.


## 4. StyleGAN: Disentangled Latent Space

**The Core Problem with Vanilla GANs:** The noise vector $Z$ controls ALL aspects of the generated image simultaneously. If you try to change someone's hair color by tweaking $Z$, their face shape and lighting might change wildly because the latent space is completely entangled. 

**StyleGAN's Solution:** 
Instead of feeding $Z$ directly to the synthesis layers, StyleGAN feeds $Z$ through an 8-layer MLP called the **Mapping Network**. This creates a disentangled intermediate latent space $W$:
$$f: Z \to W$$
Because $W$ doesn't have to follow a strict Gaussian distribution like $Z$, it can form independent, un-warped mathematical basis vectors for specific features (e.g. skin tone, gender, hair).

**Adaptive Instance Normalization (AdaIN):**
The Synthesis network processes a learned constant spatial tensor. At *every single layer*, the features are normalized, and then style from $W$ is injected using an affine transform mapping $W$ into $\gamma$ (scale) and $\beta$ (shift):
$$\text{AdaIN}(x_i, y) = y_{s,i} \frac{x_i - \mu(x)}{\sigma(x)} + y_{b,i}$$
This enforces $W$'s style information at every spatial scale unconditionally.

**Style Mixing:**
During training, StyleGAN runs two different latent codes $W_1$ and $W_2$ through the same network, switching from $W_1$ to $W_2$ at a random spatial layer. 
*Why?* It forces coarse layers (low resolution) to capture high-level structure (pose, face shape), and fine layers (high resolution) to capture texture (color, micro-details). Coarse layers can't rely on fine layers to "fix" structure, so the semantic scales perfectly decouple.


In [ ]:
# IMPLEMENTATION: Adaptive Instance Normalization (AdaIN)

class AdaIN(nn.Module):
    def __init__(self, in_channels, w_dim):
        super().__init__()
        # Map intermediate latent W to gamma (scale) and beta (shift) for each channel
        self.style_scale = nn.Linear(w_dim, in_channels)
        self.style_shift = nn.Linear(w_dim, in_channels)
        
    def forward(self, x, w):
        # x is a spatial feature map: [B, C, H, W]
        # 1. Normalize the feature map to zero mean, unit variance (Instance Norm)
        b, c, h, w_dim_spatial = x.size()
        x_view = x.view(b, c, -1)
        mean = x_view.mean(dim=2, keepdim=True).view(b, c, 1, 1)
        std = x_view.std(dim=2, keepdim=True).view(b, c, 1, 1)
        x_norm = (x - mean) / (std + 1e-8)
        
        # 2. Derive scale and shift from W
        gamma = self.style_scale(w).view(b, c, 1, 1) # [B, C, 1, 1]
        beta = self.style_shift(w).view(b, c, 1, 1)  # [B, C, 1, 1]
        
        # 3. Apply Style
        out = (x_norm * gamma) + beta
        return out

# TEST IT
w_latent = torch.randn(2, 512)            # [Batch, W_dim]
x_features = torch.randn(2, 64, 32, 32)   # [Batch, Channels, H, W]

adain_layer = AdaIN(in_channels=64, w_dim=512)
out = adain_layer(x_features, w_latent)

print(f"AdaIN Styled Feature Shape: {out.shape}")  # Should match [2, 64, 32, 32]


### COMMON PITFALLS
*   **Ignoring the W space:** When trying to build latent interpolation tools (like aging a face), practitioners often interpolate in $Z$-space. You must map $Z \to W$ for both end points and interpolate purely in the $W$ space to ensure smooth transitions without identities morphing in bizarre ways.


## 5. Latent Diffusion (Stable Diffusion Architecture)

**The Computational Problem of Pixel Denoising:** 
A standard $512 \times 512$ RGB image contains 786,432 pixels. Diffusion requires passing this massive tensor through a U-Net for 1,000 steps. 
*Compute Cost:* Running 1,000 U-Net forward passes on a $512 \times 512 \times 3$ tensor requires incredible GPU VRAM and hours per image. It is prohibitively expensive.

**The LDM Solution:**
Latent Diffusion Models (LDM) train a separate Autoencoder (VAE) to compress images into a localized, compact latent space (e.g. $64 \times 64 \times 4$ tensor). 
We perform *all* diffusion steps in this tiny latent space. The VAE Decoder is only used ONCE at the very end to convert the final denoised latent back into purely $512 \times 512$ pixel space.

**The Math (Compute Savings):**
Activations in pixel diffusion: $H \cdot W \cdot C = 512 \cdot 512 \cdot 3 = 786,432$.
Activations in latent diffusion: $64 \cdot 64 \cdot 4 = 16,384$.
Savings per step: $16,384 / 786,432 = 1/48$.
By reducing the feature map by $\sim 48\times$, the U-Net runs drastically faster and with significantly lower VRAM, while the VAE retains perceptual texture.

### Cross-Attention Conditioning
To let a text prompt "steer" the generation, clip text embeddings are injected into the U-Net's intermediate layers via Cross-Attention.
*   **Query ($Q$):** The noisy image latent features.
*   **Key ($K$) & Value ($V$):** The Text Prompt sequence embeddings.
Thus, for every spatial pixel, the model queries the text tokens to ask "What should I denoise this specific region into?"


In [ ]:
# IMPLEMENTATION: Cross-Attention Conditioning

class CrossAttentionLayer(nn.Module):
    def __init__(self, latent_dim, context_dim, embed_dim):
        super().__init__()
        self.q_proj = nn.Linear(latent_dim, embed_dim)
        self.k_proj = nn.Linear(context_dim, embed_dim)
        self.v_proj = nn.Linear(context_dim, embed_dim)
        self.scale = embed_dim ** -0.5
        
    def forward(self, x_latent, c_text):
        # x_latent: [B, Seq_img, latent_dim] (Spatial dims flattened)
        # c_text: [B, Seq_text, context_dim] 
        
        Q = self.q_proj(x_latent)  # [B, Seq_img, embed_dim]
        K = self.k_proj(c_text)    # [B, Seq_text, embed_dim]
        V = self.v_proj(c_text)    # [B, Seq_text, embed_dim]
        
        # Attention scores
        attention_logits = torch.bmm(Q, K.transpose(1, 2)) * self.scale # [B, Seq_img, Seq_text]
        attention_weights = torch.softmax(attention_logits, dim=-1)
        
        # Context-steered latents
        out = torch.bmm(attention_weights, V) # [B, Seq_img, embed_dim]
        return out

# TEST IT
B = 2
img_flattened = torch.randn(B, 64*64, 320) # A feature map from U-Net flattened
text_context = torch.randn(B, 77, 768)     # 77 CLIP tokens of dimension 768

cross_attn = CrossAttentionLayer(latent_dim=320, context_dim=768, embed_dim=320)
enhanced_img = cross_attn(img_flattened, text_context)

print(f"Cross Attn Enhanced Output Shape: {enhanced_img.shape}") # [2, 4096, 320]


### COMMON PITFALLS
*   **Relying on the U-Net for faces/text:** Because the VAE aggressively compresses images, tiny details like text fonts or distant human faces get mangled heavily inside the VAE encoder. The diffusion U-Net never sees the original pixels, so it cannot "fix" these VAE artifacts regardless of the text prompt.


## 6. Classifier-Free Guidance (CFG)

**The Problem:** Diffusion models explicitly trained to minimize MSE conditional loss generate mathematical "averages" of data that technically satisfy a text prompt but are rarely vivid, striking, or definitive.

**The Solution:** Classifier-Free Guidance (CFG) amplifies the prompt's influence without requiring an external classifier model.
During training, we randomly drop out the text condition (replace it with an empty string) 10-20% of the time. The U-Net learns both a conditioned prediction $\epsilon_\theta(x_t, c)$ and an unconditional prediction $\epsilon_\theta(x_t, \emptyset)$.

**At Inference:**
We compute both noise predictions and extrapolate AWAY from the unconditional baseline mathematically:
$$\hat{\epsilon} = \epsilon_\theta(x_t, \emptyset) + w \cdot (\epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \emptyset))$$

*   If $w = 1.0$, it is purely conditional prediction.
*   If $w = 7.5$ (Typical), the model determines the "direction" vector between unconditioned-noise and prompt-noise, and takes a massive leap in that vector's direction.

**The Tradeoff:**
$$w \uparrow \implies \text{Higher Prompt Faithfulness (CLIP), Lower Visual Diversity (FID)}$$


In [ ]:
# IMPLEMENTATION: CFG Inference Wrapper

def classifier_free_guidance_step(unet, x_t, text_cond, t, guidance_scale=7.5):
    # Create empty text conditioning (batch size must match)
    empty_cond = torch.zeros_like(text_cond) 
    
    # 1. Unconditional Prediction
    noise_uncond = unet(x_t, empty_cond, t)
    
    # 2. Conditional Prediction
    noise_cond = unet(x_t, text_cond, t)
    
    # 3. Vector Extrapolation
    noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
    
    return noise_pred

# TEST IT
# Dummy U-Net proxy
class DummyUNet(nn.Module):
    def forward(self, x, c, t):
        # We simulate the unet using context to shift noise
        return x * 0.5 + c.mean(dim=(1,2)).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1) * 0.1

B, C, H, W = 1, 4, 64, 64
x_t = torch.randn(B, C, H, W)
text_c = torch.randn(B, 77, 768)  # Our conditional prompt
t = torch.tensor([500])

unet = DummyUNet()
guided_noise = classifier_free_guidance_step(unet, x_t, text_c, t, guidance_scale=7.5)
print(f"Guided Noise Prediction Shape: {guided_noise.shape}")


### COMMON PITFALLS
*   **Burn-out Artifacts:** Setting CFG too high ($w > 15$) causes the extrapolation to shoot past the boundaries of the image manifold. Saturated colors, extreme contrast, and harsh artifacts appear.


## 7. DDIM: Deterministic Sampling

**The Problem with DDPM:** DDPM is a strict Markov chain. If it took 1000 steps of adding Gaussian noise to blur an image, it mathematically demands exactly 1000 steps of U-Net deduction to denoise it. At 50ms per step, rendering one image takes $50$ seconds.

**DDIM (Denoising Diffusion Implicit Models):**
DDIM mathematically rewrites the diffusion marginals as a *non-Markovian* process. It proves that the forward process isn't strictly chained; you just need to know the initial image $x_0$ and the noise. Because the sampling equations are now deterministic (no random variance term $\sigma_t$ added at each backward step), we can "skip" steps in the trajectory.
Instead of: $T = 1000 \to 999 \to 998 \dots$
We can jump: $T = 1000 \to 950 \to 900 \dots$

If we use DDIM with 50 steps, we do 50 U-Net passes. $50 \times 50\text{ms} = 2.5$ seconds instead of 50. Visually, the output is nearly identical.


In [ ]:
# IMPLEMENTATION: Conceptual DDIM Step Formulation

def ddim_step(x_t, noise_pred, t, previous_t, alpha_cumprod):
    # This represents mathematically skipping the intermediate Markov links.
    # Get alpha_bars
    alpha_bar_t = alpha_cumprod[t]
    alpha_bar_prev = alpha_cumprod[previous_t]
    
    # 1. Predict x_0 from the current noise assessment
    pred_x0 = (x_t - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)
    
    # 2. Point to the previous deterministic timestep (No random noise added)
    dir_xt = torch.sqrt(1 - alpha_bar_prev) * noise_pred
    
    # 3. Compile x_{t-1}
    x_prev = torch.sqrt(alpha_bar_prev) * pred_x0 + dir_xt
    return x_prev

# TEST IT
# Assume we have precalculated Alpha schedules (Cosine/Linear)
T = 1000
alpha_cumprod = torch.linspace(1.0, 0.0, T + 1)  # Simplified schedule
x_t = torch.randn(1, 4, 64, 64)
noise_p = torch.randn(1, 4, 64, 64) # Pseudo U-Net output

# Jump 50 steps!
x_prev = ddim_step(x_t, noise_p, t=1000, previous_t=950, alpha_cumprod=alpha_cumprod)
print(f"Determinisitc DDIM jumped inference shape: {x_prev.shape}")


### COMMON PITFALLS
*   **Attempting 5 steps:** While DDIM scales down to 50 or 20 steps, reducing it to 5 steps heavily degrades quality because the curved generative trajectory is approximated poorly by huge linear steps. (Flow-matching rectifies this in modern models).


## 8. Evaluation Metrics for Generative Models

How do we mathematically prove a Generative Model is good? By treating it as a statistical distribution problem comparing Real (r) datasets against Generated (g) datasets.

1.  **FID (Fréchet Inception Distance):** 
    Computes the Mean ($\mu$) and Covariance ($\Sigma$) of deep Inception-v3 features for real and generated images. 
    $$\text{FID} = ||\mu_r - \mu_g||^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$
    *Lower is better.* FID=0 means the real and generated data distributions perfectly overlap. Highly sensitive to Mode Collapse (Covariance $\Sigma_g$ vanishes).
2.  **LPIPS (Learned Perceptual Image Patch Similarity):**
    A perceptual loss mapping images through VGG networks, penalizing differences in deep semantic features rather than pixels. Correlates flawlessly with human judgment.
3.  **CLIP Score:**
    Cosine similarity between the CLIP text embedding of the Prompt and the CLIP image embedding of the final generation. Evaluates alignment/faithfulness.


In [ ]:
# IMPLEMENTATION: FID Calculation Math
import scipy.linalg
import numpy as np

def calculate_frechet_distance(mu_r, sigma_r, mu_g, sigma_g):
    # 1. Squared euclidean distance of means
    diff = mu_r - mu_g
    
    # 2. Covariance product matrix square root
    covmean, _ = scipy.linalg.sqrtm(sigma_r.dot(sigma_g), disp=False)
    
    # Handle numerical inaccuracies causing imaginary components
    if np.iscomplexobj(covmean):
        covmean = covmean.real
        
    # 3. Trace calculation
    tr_covmean = np.trace(covmean)
    
    return (diff.dot(diff) + np.trace(sigma_r) + np.trace(sigma_g) - 2 * tr_covmean)

# TEST IT
# Synthesize inception features for Real (10k images, 2048 dims)
mu_real = np.ones(2048) * 0.5
sigma_real = np.eye(2048) * 1.5

# Gen faces Mode Collapse simulation (Variance is tiny)
mu_gen = np.ones(2048) * 0.4
sigma_gen = np.eye(2048) * 0.1

fid_score = calculate_frechet_distance(mu_real, sigma_real, mu_gen, sigma_gen)
print(f"Calculated FID Score: {fid_score:.2f}")
# A high FID proves the Generator lacks structural variance compared to reality.


### COMMON PITFALLS
*   **Sample Size Discrepancies:** FID is biased by sample size. Calculating FID between 5k real and 5k generated images yields radically different baseline numbers than 50k vs 50k. Always standardize sample counts (e.g. 30k) when comparing papers.
